# 04 — Estimate Facility Capacity

## Research purpose

Combine OSM geometry with available Overture building evidence to estimate facility IT load and total power. These values are standardised capacity proxies rather than reported capacities.


## Imports

Load the libraries used to read source records, process geometry and acquire missing Overture extracts.


In [1]:
import ast
import json
import math
from pathlib import Path
import re
import shutil
import subprocess

import geopandas as gpd
import pandas as pd

## Project location and paths

Identify the project directory and define the paths for the grouped facilities, Overture extracts and OSM building footprints.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

geojson_path = DATA_RAW / "overture" / "overture_buildings_raw.geojson"
csv_path = DATA_RAW / "overture" / "overture_building_heights_raw.csv"
chunk_dir = DATA_RAW / "overture" / "overture_buildings_by_campus"
building_path = DATA_RAW / "osm" / "osm_building_footprints_raw.geojson"

## Capacity and acquisition settings

Set the IT-space share to 0.60 and the central power density to 3,000 W/m², with sensitivity cases of 1,000 and 5,000 W/m². Use PUE values of 1.09 for Google, 1.15 for Amazon and 1.17 for Microsoft, with 1.15 as the fallback. Missing Overture extracts use release `2026-04-15.0` and campus bounding boxes padded by 250 m.


In [3]:
DEFAULT_FLOORS = 1.0
TECHNICAL_FLOOR_HEIGHT_M = 6.0
IT_SPACE_SHARE = 0.60
IT_POWER_DENSITY_W_PER_M2 = 3000.0
IT_POWER_DENSITY_SENSITIVITY_W_PER_M2 = {
    "low": 1000.0,
    "central": IT_POWER_DENSITY_W_PER_M2,
    "high": 5000.0,
}

OPERATOR_PUE = {
    "Google": 1.09, # https://datacenters.google/efficiency/
    "Amazon": 1.15, # https://aws.amazon.com/sustainability/data-centers/
    "Microsoft": 1.17, # https://datacenters.microsoft.com/sustainability/efficiency/
}

FALLBACK_PUE = 1.15

# Acquisition controls
FETCH_OVERTURE_IF_MISSING = True
OVERTURE_RELEASE = "2026-04-15.0"
OVERTURE_BBOX_PADDING_M = 250
OVERTURE_DOWNLOAD_FORMAT = "geojson"

## Calculate capacity from floorspace

Define gross floor area as building footprint × floors and IT load in MW as gross floor area × 0.60 × 3,000 W/m² ÷ 1,000,000. Calculate total power by multiplying IT load by the operator PUE. The function leaves capacity missing when no usable footprint is available and accepts alternative power densities for the sensitivity analysis.


In [4]:
def derive_capacity_mw(
    building_footprint_area_m2: float | None,
    floors: float | None = None,
    operator_group: str | None = None,
    power_density_w_per_m2: float = IT_POWER_DENSITY_W_PER_M2,
) -> dict[str, float | str | None]:
    if building_footprint_area_m2 is None or building_footprint_area_m2 <= 0:
        return {
            "gross_floor_area_m2": None,
            "it_load_mw": None,
            "assumed_pue": OPERATOR_PUE.get(operator_group or "", FALLBACK_PUE),
            "capacity_mw": None,
            "capacity_estimation_method": "not_available",
            "capacity_confidence": "unknown",
        }
    floors = floors if floors and floors > 0 else DEFAULT_FLOORS
    gross_floor_area_m2 = building_footprint_area_m2 * floors
    it_load_mw = gross_floor_area_m2 * IT_SPACE_SHARE * power_density_w_per_m2 / 1_000_000
    pue = OPERATOR_PUE.get(operator_group or "", FALLBACK_PUE)
    return {
        "gross_floor_area_m2": gross_floor_area_m2,
        "it_load_mw": it_load_mw,
        "assumed_pue": pue,
        "capacity_mw": it_load_mw * pue,
        "capacity_estimation_method": "footprint_x_floors_x_it_space_share_x_power_density",
        "capacity_confidence": "medium" if floors != DEFAULT_FLOORS else "low",
    }

## Load grouped facilities

Load the grouped facility table and check that it contains the OSM identifiers, campus groups, capacity roles and operators needed for evidence matching.


In [5]:
facilities = pd.read_csv(DATA_PROCESSED / "datacenters_hyperscalers_deduplicated.csv")
required = {"osm_ref", "osm_id", "osm_type", "campus_group_id", "capacity_record_role", "operator_group"}
missing = sorted(required - set(facilities.columns))

if missing:
    raise ValueError(f"Grouped facility input is missing {missing}")

print(f"Grouped facility rows: {len(facilities):,}")

Grouped facility rows: 511


## Acquire missing Overture extracts

Define functions to create padded campus bounding boxes and acquire building data from the pinned Overture release. Existing campus extracts are reused, missing extracts are downloaded with the official tool, and the features are combined by identifier.


In [6]:
# Create a padded longitude/latitude box around a campus for Overture acquisition.
def bbox_for_group(group: pd.DataFrame, padding_m: float = 250) -> tuple[float, float, float, float]:
    lat_min = float(group["latitude"].min())
    lat_max = float(group["latitude"].max())
    lon_min = float(group["longitude"].min())
    lon_max = float(group["longitude"].max())
    lat_mid = (lat_min + lat_max) / 2
    lat_pad = padding_m / 111_320
    lon_pad = padding_m / (111_320 * max(math.cos(math.radians(lat_mid)), 0.2))
    return lon_min - lon_pad, lat_min - lat_pad, lon_max + lon_pad, lat_max + lat_pad


# Reuse campus chunks, acquire missing chunks, and assemble their building features.
def fetch_overture_buildings_by_campus() -> None:
    overture_cli = None
    chunk_dir.mkdir(parents=True, exist_ok=True)
    groups = list(facilities.groupby("campus_group_id", dropna=False))

    chunk_paths = []
    for number, (campus_group_id, group) in enumerate(groups, start=1):
        bbox = bbox_for_group(group, padding_m=OVERTURE_BBOX_PADDING_M)
        campus_label = str(campus_group_id) if pd.notna(campus_group_id) else f"ungrouped_{number:05d}"
        out_path = chunk_dir / f"{campus_label}.geojson"
        if not out_path.exists():
            if overture_cli is None:
                overture_cli = shutil.which("overturemaps")
                if overture_cli is None:
                    raise RuntimeError(
                        f"Missing Overture chunk {out_path}. The overturemaps CLI is required "
                        "only to acquire missing chunks. Install the documented environment "
                        "or provide the pinned-release campus extract."
                    )
            cmd = [
                overture_cli,
                "download",
                "--release", OVERTURE_RELEASE,
                "--bbox", ",".join(f"{value:.7f}" for value in bbox),
                "-f", OVERTURE_DOWNLOAD_FORMAT,
                "--type", "building",
                "-o", str(out_path),
            ]
            print(f"Downloading {campus_label}: {cmd[3]}")
            subprocess.run(cmd, check=True)
        if out_path.exists() and out_path.stat().st_size > 0:
            chunk_paths.append(out_path)

    features_by_id = {}
    for path in chunk_paths:
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            for feature in data.get("features", []):
                feature_id = feature.get("id") or feature.get("properties", {}).get("id") or json.dumps(feature.get("geometry"), sort_keys=True)
                features_by_id[feature_id] = feature
        except Exception as exc:
            print(f"Could not read {path}: {exc}")
    if not features_by_id:
        print("No Overture buildings were downloaded.")
        return
    combined = {
        "type": "FeatureCollection",
        "features": list(features_by_id.values()),
    }
    geojson_path.write_text(json.dumps(combined), encoding="utf-8")
    print(f"Saved combined Overture extract: {geojson_path} ({len(features_by_id):,} rows)")

## Load Overture evidence

Load the available Overture GeoJSON or CSV and retain the source-file path for each record. If neither combined file exists and acquisition is enabled, assemble the campus extracts and download any missing ones.


In [7]:
# Read feature properties and retain their Overture ID and input-file provenance.
def rows_from_overture_geojson(path: Path) -> pd.DataFrame:
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for feature in data.get("features", []):
        props = dict(feature.get("properties", {}) or {})
        props["id"] = feature.get("id") or props.get("id")
        props["_overture_input_file"] = str(path)
        rows.append(props)
    return pd.DataFrame(rows)

# Load the available combined extract, acquiring missing campus evidence when needed.
if not geojson_path.exists() and not csv_path.exists() and FETCH_OVERTURE_IF_MISSING:
    fetch_overture_buildings_by_campus()
if geojson_path.exists():
    overture = rows_from_overture_geojson(geojson_path)
elif csv_path.exists():
    overture = pd.read_csv(csv_path)
    overture["_overture_input_file"] = str(csv_path)
else:
    overture = pd.DataFrame()

print(f"Overture rows loaded: {len(overture):,}")

if not overture.empty:
    display(overture.head())

Overture rows loaded: 15,485


,id,names,sources,level,height,is_underground,num_floors,subtype,class,roof_material,...,min_floor,facade_color,facade_material,roof_shape,roof_color,roof_height,num_floors_underground,roof_orientation,roof_direction,_overture_input_file
0,78c2667a-e89e-409c-a914-a66ab4fe7c93,"{'primary': 'Tacos & Tequilas Mexican Grill', ...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,10.000000,False,2.0,commercial,None,None,...,NaN,None,None,None,None,NaN,NaN,None,NaN,/Users/stefan/Developer/final test/data/raw/ov...
1,733678ad-95ae-4dd1-b9d7-e109ad029db8,"{'primary': 'Eats', 'common': None, 'rules': N...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,3.770254,False,NaN,commercial,None,None,...,NaN,None,None,None,None,NaN,NaN,None,NaN,/Users/stefan/Developer/final test/data/raw/ov...
2,521a7294-ce90-45d0-a3bc-b270c2bc3446,"{'primary': 'Tech Genius', 'common': None, 'ru...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,4.133075,False,NaN,None,None,None,...,NaN,None,None,None,None,NaN,NaN,None,NaN,/Users/stefan/Developer/final test/data/raw/ov...
3,4f3b1a32-e4c0-4d63-a69d-f8a06591f57d,None,"[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,NaN,False,NaN,None,None,None,...,NaN,None,None,None,None,NaN,NaN,None,NaN,/Users/stefan/Developer/final test/data/raw/ov...
4,efe344b4-0bb2-4ec0-9433-eaab7a004c1d,"{'primary': 'Mister Hot Shine Car Wash', 'comm...","[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,7.200000,False,NaN,None,None,None,...,NaN,None,None,None,None,NaN,NaN,None,NaN,/Users/stefan/Developer/final test/data/raw/ov...


## Prepare source identifiers

Define functions to standardise OSM references and extract matching identifiers from the Overture source fields. Define how height and floor-count values are selected and converted to numbers.


In [8]:
OSM_PREFIX = {"node": "n", "way": "w", "relation": "r"}

# Convert the OSM element type and numeric ID to an Overture-compatible source key.
def osm_join_key(row):
    prefix = OSM_PREFIX.get(str(row.get("osm_type", "")).lower())
    osm_id = row.get("osm_id")
    if not prefix or pd.isna(osm_id):
        return None
    try:
        return f"{prefix}{int(float(osm_id))}"
    except Exception:
        return None

# Extract the numeric token used by the existing height and floor evidence parser.
def parse_numeric(value):
    if pd.isna(value):
        return None
    match = re.search(r"[0-9]+(?:\.[0-9]+)?", str(value))
    if not match:
        return None
    return float(match.group(0))

# Normalize source-record references to comparable OSM identity keys.
def normalize_record_id(value):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    if not value:
        return None
    value = value.split("@", 1)[0]
    match = re.search(r"[nwr][0-9]+", value)
    return match.group(0) if match else value

# Collect unique source keys from the supported flat and nested Overture fields.
def extract_source_record_ids(row):
    candidate_columns = [
        "source_record_id", "record_id", "source_id", "sources.record_id",
        "sources_0_record_id", "sources:record_id",
    ]
    ids = []
    for column in candidate_columns:
        if column in row.index:
            key = normalize_record_id(row.get(column))
            if key:
                ids.append(key)
    if "sources" in row.index:
        raw = row.get("sources")
        parsed = None
        if isinstance(raw, (list, dict)):
            parsed = raw
        elif raw is not None and not (isinstance(raw, float) and pd.isna(raw)) and str(raw).strip():
            try:
                parsed = json.loads(str(raw))
            except Exception:
                try:
                    parsed = ast.literal_eval(str(raw))
                except Exception:
                    parsed = None
        source_items = parsed if isinstance(parsed, list) else [parsed] if isinstance(parsed, dict) else []
        for item in source_items:
            if not isinstance(item, dict):
                continue
            for key_name in ["record_id", "source_record_id", "id"]:
                key = normalize_record_id(item.get(key_name))
                if key:
                    ids.append(key)
    return sorted(set(ids))

# Choose the first present value among the supported alternative evidence columns.
def first_existing(row, names):
    for name in names:
        if name in row.index and pd.notna(row.get(name)):
            return row.get(name)
    return None

## Match Overture records

Build a table of Overture height and floor evidence keyed by OSM source identifier. When several records share a key, sort by ascending height and floor count, place missing values last and keep the first record.


In [9]:
facilities["overture_osm_join_key"] = facilities.apply(osm_join_key, axis=1)

overture_matches = []
if not overture.empty:
    for _, row in overture.iterrows():
        source_ids = extract_source_record_ids(row)
        height = parse_numeric(first_existing(row, ["height", "height_m", "building_height", "overture_height_m"]))
        num_floors = parse_numeric(first_existing(row, ["num_floors", "floors", "floor_count", "overture_num_floors"]))
        for source_id in source_ids:
            overture_matches.append({
                "overture_osm_join_key": source_id,
                "overture_height_m": height,
                "overture_num_floors": num_floors,
                "overture_source_record_id": source_id,
                "overture_feature_id": row.get("id"),
                "overture_input_file": row.get("_overture_input_file"),
            })

overture_match_df = pd.DataFrame(overture_matches)
if not overture_match_df.empty:
    overture_match_df = overture_match_df.sort_values(
        ["overture_osm_join_key", "overture_height_m", "overture_num_floors"],
        na_position="last",
    ).drop_duplicates("overture_osm_join_key", keep="first")

print(f"Overture source IDs extracted: {len(overture_match_df):,}")
overture_match_df.head() if not overture_match_df.empty else None

Overture source IDs extracted: 12,798


,overture_osm_join_key,overture_height_m,overture_num_floors,overture_source_record_id,overture_feature_id,overture_input_file
8509,003B0B4D-99DF-431F-BF32-F3FD74766D0E,NaN,NaN,003B0B4D-99DF-431F-BF32-F3FD74766D0E,0594ddc6-1363-42be-9a8e-17d47f4ea4ec,/Users/stefan/Developer/final test/data/raw/ov...
9815,008F136D-53DE-46C2-BFDF-41D72A27B688,4.0,NaN,008F136D-53DE-46C2-BFDF-41D72A27B688,5e3d6905-b88a-48eb-9e38-c740538de704,/Users/stefan/Developer/final test/data/raw/ov...
1068,00A6E7B3-9016-451F-9AB4-856089BC20D8,NaN,NaN,00A6E7B3-9016-451F-9AB4-856089BC20D8,d1478c99-14ef-4634-adb1-31330567ce93,/Users/stefan/Developer/final test/data/raw/ov...
9574,00E432EB-FE71-4C80-95D3-6793C0A27509,4.6,NaN,00E432EB-FE71-4C80-95D3-6793C0A27509,4d6354a5-655a-4ea2-b509-cc33a92ba44e,/Users/stefan/Developer/final test/data/raw/ov...
9709,023035A7-F730-4724-90ED-BDB138527E3B,4.6,NaN,023035A7-F730-4724-90ED-BDB138527E3B,dc24e45e-e366-4972-a76b-b8e8c2c35848,/Users/stefan/Developer/final test/data/raw/ov...


## Save matched height evidence

Join the selected Overture evidence to the facility table and leave unmatched values missing. Save the enriched table and a separate review of records without an Overture match, excluding context-only sites from that review.


In [10]:
if overture_match_df.empty:
    enriched = facilities.copy()
    for column in [
        "overture_height_m", "overture_num_floors", "overture_source_record_id",
        "overture_feature_id", "overture_input_file",
    ]:
        enriched[column] = pd.NA
else:
    enriched = facilities.merge(overture_match_df, on="overture_osm_join_key", how="left")

matched = enriched["overture_source_record_id"].notna()
review = enriched[
    enriched["capacity_record_role"].ne("site_context_only") & ~matched
].copy()

enriched.to_csv(DATA_PROCESSED / "datacenters_hyperscalers_height_enriched.csv", index=False)
review.to_csv(DATA_INTERIM / "datacenters_overture_height_review.csv", index=False)

print(f"Height-enriched rows: {len(enriched):,}")
print(f"Matched Overture source identifiers: {int(matched.sum()):,}")
print(f"Rows needing Overture height review: {len(review):,}")
enriched[["facility_name_raw", "operator_group", "osm_ref", "overture_height_m", "overture_num_floors"]].head()

Height-enriched rows: 511
Matched Overture source identifiers: 436
Rows needing Overture height review: 56


,facility_name_raw,operator_group,osm_ref,overture_height_m,overture_num_floors
0,Google Fiber,Google,node/4879481820,NaN,NaN
1,Amazon SIN4,Amazon,node/8208591051,NaN,NaN
2,Amazon SIN2,Amazon,node/8208591054,NaN,NaN
3,Google Singapore DC,Google,node/8208591112,NaN,NaN
4,Amazon SIN3,Amazon,node/8208598531,NaN,NaN


## Load geometry evidence

Prepare the matched height and floor values and load the normalised OSM geometries. Load the context-building snapshot when available and display the number of records in each evidence layer.


In [11]:
facilities = enriched.copy()
for column in ["overture_height_m", "overture_num_floors"]:
    facilities[column] = pd.to_numeric(
        facilities[column].astype("string"), errors="coerce"
    ).astype("float64")
dc_gdf = gpd.read_file(DATA_INTERIM / "osm_datacenters_geometry_normalized.geojson")
if building_path.exists():
    buildings_gdf = gpd.read_file(building_path)
else:
    buildings_gdf = gpd.GeoDataFrame(columns=["source_id", "osm_ref", "context_source_id", "polygon_area_m2"], geometry=[], crs="EPSG:4326")

for frame in [dc_gdf, buildings_gdf]:
    if not frame.empty and frame.crs is None:
        frame.set_crs("EPSG:4326", inplace=True)

print(f"Facilities: {len(facilities):,}")
print(f"OSM data-centre geometries: {len(dc_gdf):,}")
print(f"Context buildings: {len(buildings_gdf):,}")

Facilities: 511
OSM data-centre geometries: 4,286
Context buildings: 34,810


## Measure building proximity

Define the great-circle distance calculation used to match point facilities with nearby building footprints. Distances are returned in metres.


In [12]:
def haversine_m(lat1, lon1, lat2, lon2):
    radius = 6371008.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))

## Resolve floor counts

Create an OSM geometry lookup and define the floor-count evidence hierarchy. Prefer OSM building levels, followed by Overture floor counts, Overture height and then OSM height, converting height to floors by dividing by 6 m and rounding. Use one floor when no usable evidence is available and retain the selected method and source.


In [13]:
dc_lookup = {row["osm_ref"]: row for _, row in dc_gdf.iterrows() if pd.notna(row.get("osm_ref"))}

# Extract the positive numeric token used by the existing floor-evidence parser.
def parse_positive_float(value):
    if pd.isna(value):
        return None
    match = re.search(r"[0-9]+(?:\.[0-9]+)?", str(value))
    if not match:
        return None
    number = float(match.group(0))
    return number if number > 0 else None

# Apply the declared floor-count evidence hierarchy and return its provenance.
def floors_from_tags(row):
    levels = parse_positive_float(row.get("building_levels")) or parse_positive_float(row.get("levels_raw"))
    if levels:
        return levels, "osm_building_levels", "high", parse_positive_float(row.get("height")) or parse_positive_float(row.get("height_raw")), "OpenStreetMap"

    overture_num_floors = parse_positive_float(row.get("overture_num_floors"))
    if overture_num_floors:
        return overture_num_floors, "overture_num_floors", "medium", parse_positive_float(row.get("overture_height_m")), "Overture Maps"

    overture_height = parse_positive_float(row.get("overture_height_m")) or parse_positive_float(row.get("height_m"))
    if overture_height:
        floors = float(max(round(overture_height / TECHNICAL_FLOOR_HEIGHT_M), 1))
        return floors, f"overture_height_divided_by_{TECHNICAL_FLOOR_HEIGHT_M:g}m_rounded", "medium", overture_height, "Overture Maps"

    osm_height = parse_positive_float(row.get("height")) or parse_positive_float(row.get("height_raw"))
    if osm_height:
        floors = float(max(round(osm_height / TECHNICAL_FLOOR_HEIGHT_M), 1))
        return floors, f"osm_height_divided_by_{TECHNICAL_FLOOR_HEIGHT_M:g}m_rounded", "low", osm_height, "OpenStreetMap"

    return DEFAULT_FLOORS, "default_one_floor_assumption", "low", None, pd.NA

## Resolve building footprints

Retain the site-area exceptions for AWS Querétaro (`way/1448056077`) and AWS New Zealand (`way/1088665595`), because their building footprints were not mapped. These exceptions allow site area as a low-confidence footprint proxy only when building evidence is missing and the record is not context-only. Define functions to select context buildings within site boundaries or within 150 m of point facilities, prioritising their source-record associations.


In [14]:
# Reviewed exceptions: use site area only when no building footprint is available.
approved_proxy_reasons = {
    "way/1448056077": "Amazon AWS Queretaro recent site; building footprint not yet mapped",
    "way/1088665595": "Amazon Web Services New Zealand site; building footprint not yet mapped",
}
SITE_AREA_PROXY_METHOD = "manual_site_area_proxy_recent_or_unmapped_building"

# Look up an explicitly documented site-area proxy exception by OSM reference.
def site_area_proxy_reason(refs):
    for ref in refs:
        if ref in approved_proxy_reasons:
            return approved_proxy_reasons[ref]
    return None

# Select context buildings whose representative points lie within the site geometry.
def buildings_inside_site(site_geometry, context_source_id):
    if buildings_gdf.empty or site_geometry is None or site_geometry.is_empty:
        return buildings_gdf.iloc[0:0].copy()
    candidates = buildings_gdf
    if "context_source_id" in candidates.columns and context_source_id:
        direct = candidates[candidates["context_source_id"] == context_source_id]
        if not direct.empty:
            candidates = direct
    reps = candidates.geometry.representative_point()
    return candidates[reps.within(site_geometry)].copy()

# Select context buildings within the point-facility search radius.
def nearby_buildings(lat, lon, context_source_id=None, radius_m=150):
    if buildings_gdf.empty or pd.isna(lat) or pd.isna(lon):
        return buildings_gdf.iloc[0:0].copy()
    candidates = buildings_gdf
    if "context_source_id" in candidates.columns and context_source_id:
        direct = candidates[candidates["context_source_id"] == context_source_id]
        if not direct.empty:
            candidates = direct
    distances = candidates.geometry.representative_point().apply(lambda p: haversine_m(float(lat), float(lon), p.y, p.x))
    return candidates[distances <= radius_m].copy()

## Estimate facility capacity

Resolve each facility’s footprint and floor count, retain the evidence sources and apply the capacity formula. Use the two site-area exceptions only when their conditions are met and leave independent capacity missing for context-only sites. Collect records with missing footprint evidence for review.


In [15]:
estimate_rows = []
review_rows = []

for _, facility in facilities.iterrows():
    refs = [f"{facility['osm_type']}/{int(facility['osm_id'])}"]
    used_building_refs = set()
    footprint_area = 0.0
    site_area = 0.0
    methods = []
    confidence = "unknown"
    needs_capacity_review = False
    floors, floor_method, floor_confidence, height_m, height_source = floors_from_tags(facility)
    record_role = facility.get("capacity_record_role", "facility_candidate")

    for ref in refs:
        source_row = dc_lookup.get(ref)
        if source_row is None:
            continue
        geom = source_row.geometry
        geom_kind = source_row.get("geometry_kind")
        area = pd.to_numeric(source_row.get("polygon_area_m2"), errors="coerce")
        source_id = source_row.get("source_id")

        if record_role == "site_context_only":
            if pd.notna(area) and geom_kind in {"site_or_landuse_polygon", "unclassified_polygon"}:
                site_area += float(area)
            continue

        if geom_kind == "building_footprint" and pd.notna(area) and area > 0:
            if ref not in used_building_refs:
                footprint_area += float(area)
                used_building_refs.add(ref)
                methods.append("direct_osm_building_polygon")
                confidence = "medium"
        elif geom_kind in {"site_or_landuse_polygon", "unclassified_polygon"}:
            if pd.notna(area):
                site_area += float(area)
            inside = buildings_inside_site(geom, source_id)
            for _, building in inside.iterrows():
                building_ref = building.get("osm_ref")
                building_area = pd.to_numeric(building.get("polygon_area_m2"), errors="coerce")
                if pd.notna(building_area) and building_area > 0 and building_ref not in used_building_refs:
                    footprint_area += float(building_area)
                    used_building_refs.add(building_ref)
            if not inside.empty:
                methods.append("context_buildings_inside_osm_site_polygon")
                confidence = "low" if confidence == "unknown" else confidence
            else:
                methods.append("site_polygon_no_building_footprints")
                needs_capacity_review = True
        elif geom_kind == "point":
            nearby = nearby_buildings(facility.get("latitude"), facility.get("longitude"), source_id)
            for _, building in nearby.iterrows():
                building_ref = building.get("osm_ref")
                building_area = pd.to_numeric(building.get("polygon_area_m2"), errors="coerce")
                if pd.notna(building_area) and building_area > 0 and building_ref not in used_building_refs:
                    footprint_area += float(building_area)
                    used_building_refs.add(building_ref)
            if not nearby.empty:
                methods.append("context_buildings_within_150m_of_osm_node")
                confidence = "low"
            else:
                methods.append("node_no_building_footprints")
                needs_capacity_review = True

    footprint_area_value = footprint_area if footprint_area > 0 else None
    site_area_value = site_area if site_area > 0 else None
    site_proxy_reason = site_area_proxy_reason(refs)
    used_site_area_proxy = False

    if (
        footprint_area_value is None
        and site_area_value is not None
        and record_role != "site_context_only"
        and site_proxy_reason is not None
    ):
        footprint_area_value = site_area_value
        used_site_area_proxy = True
        methods = [SITE_AREA_PROXY_METHOD]
        confidence = "low"
        needs_capacity_review = False

    estimates = derive_capacity_mw(footprint_area_value, floors=floors, operator_group=facility.get("operator_group"))
    if used_site_area_proxy:
        estimates.update({
            "capacity_estimation_method": "manual_site_area_proxy_x_floors_x_it_space_share_x_power_density",
            "capacity_confidence": "low",
        })
    if record_role == "site_context_only":
        estimates.update({
            "gross_floor_area_m2": None,
            "it_load_mw": None,
            "capacity_mw": None,
            "capacity_estimation_method": "site_context_only_not_capacity_bearing",
            "capacity_confidence": "unknown",
        })
        methods = ["site_context_only_not_capacity_bearing"]
        confidence = "not_applicable"
    # Overture inputs have served the floor selection; keep its selected evidence below.
    output = facility.drop(labels=[
        "overture_osm_join_key", "overture_height_m", "overture_num_floors",
        "overture_source_record_id", "overture_feature_id", "overture_input_file",
    ]).to_dict()
    output.update(estimates)
    existing_notes = "" if pd.isna(output.get("notes", "")) else str(output.get("notes", ""))
    output["notes"] = existing_notes
    if used_site_area_proxy:
        output["notes"] = (
            existing_notes
            + " Manual site-area capacity proxy accepted after facility review: "
            + site_proxy_reason
            + "; no OSM building footprint was available at run time."
        ).strip()
    output.update({
        "site_area_m2": site_area_value,
        "building_footprint_area_m2": footprint_area_value,
        "floors_assumed": floors,
        "floor_count_method": floor_method,
        "floor_count_confidence": floor_confidence,
        "height_m": height_m,
        "height_source": height_source,
        "footprint_method": ";".join(sorted(set(methods))) if methods else "not_available",
        "footprint_confidence": confidence,
        "footprint_source_refs": "|".join(sorted(used_building_refs)),
        "used_site_area_capacity_proxy": used_site_area_proxy,
        "site_area_capacity_proxy_reason": site_proxy_reason if used_site_area_proxy else pd.NA,
    })
    needs_capacity_review = needs_capacity_review or (
        footprint_area_value is None and record_role != "site_context_only"
    )
    if needs_capacity_review:
        output["notes"] = (
            output["notes"] + " Building footprint evidence is missing or incomplete; "
            "inspect the footprint method before using this capacity estimate."
        ).strip()
        review_rows.append(output)
    estimate_rows.append(output)

## Save facility estimates

Save the facility estimates with `it_load_mw` for IT load and `capacity_mw` for total power. Save the evidence-review table and display a sample of the estimates.


In [16]:
estimated = pd.DataFrame(estimate_rows)
review = pd.DataFrame(review_rows, columns=estimated.columns)

estimated.to_csv(DATA_PROCESSED / "datacenters_hyperscalers_capacity_estimated.csv", index=False)
review.to_csv(DATA_INTERIM / "datacenters_capacity_review.csv", index=False)

print(estimated[["operator_group", "capacity_record_role", "building_footprint_area_m2", "it_load_mw", "capacity_mw"]].head())
print(f"Rows requiring review: {len(review):,}")

  operator_group  capacity_record_role  building_footprint_area_m2  \
0         Google  point_facility_proxy                23840.669246   
1         Amazon  point_facility_proxy                29779.322319   
2         Amazon  point_facility_proxy                         NaN   
3         Google  point_facility_proxy                21849.379085   
4         Amazon  point_facility_proxy                19205.538387   

   it_load_mw  capacity_mw  
0   42.913205    46.775393  
1   53.602780    61.643197  
2         NaN          NaN  
3   39.328882    42.868482  
4   34.569969    39.755464  
Rows requiring review: 3


## Capacity-density sensitivity

Recalculate IT load and total power for the low, central and high power-density assumptions, keeping each facility’s footprint, floors and operator PUE fixed. Exclude context-only records and unusable footprints and save the sensitivity table.


In [17]:
sensitivity_rows = []

for _, row in estimated.iterrows():
    footprint_area = pd.to_numeric(row.get("building_footprint_area_m2"), errors="coerce")
    floors = pd.to_numeric(row.get("floors_assumed"), errors="coerce")
    record_role = row.get("capacity_record_role", "facility_candidate")

    if record_role == "site_context_only" or pd.isna(footprint_area) or footprint_area <= 0:
        continue

    for case, density in IT_POWER_DENSITY_SENSITIVITY_W_PER_M2.items():
        estimates = derive_capacity_mw(
            float(footprint_area),
            floors=float(floors) if pd.notna(floors) and floors > 0 else DEFAULT_FLOORS,
            operator_group=row.get("operator_group"),
            power_density_w_per_m2=float(density),
        )
        sensitivity_rows.append({
            "facility_id_source": row.get("source_ids") or row.get("osm_ref") or row.get("osm_id"),
            "operator_group": row.get("operator_group"),
            "country_code": row.get("country_code"),
            "timezone": row.get("timezone"),
            "campus_group_id": row.get("campus_group_id"),
            "capacity_record_role": record_role,
            "it_density_case": case,
            "it_power_density_w_per_m2": density,
            "gross_floor_area_m2": estimates["gross_floor_area_m2"],
            "it_load_mw": estimates["it_load_mw"],
            "capacity_mw": estimates["capacity_mw"],
            "assumed_pue": estimates["assumed_pue"],
        })

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(DATA_INTERIM / "datacenters_capacity_sensitivity.csv", index=False)
print(f"Sensitivity rows: {len(sensitivity):,}")

Sensitivity rows: 1,467
